# 10 · How the calibration map is built: option A or B

**Question:** Verra's tests need, for each model, the loss rate of each of its risk classes. Which map should those rates be learned on? The answer decides which model leads in notebook 08. This notebook sets out what the two options mean, what Verra's documents and the tools say, whether riskmapjnr's lead under option B holds with other scores, and whether the improved model can do better.

**How Verra's tests use a risk map** (notebook 03). Take the first window: calibration 2014–2016 (from T1, the end of 2013, to T2, the end of 2016) and confirmation 2017–2019 (T2 to T3, the end of 2019).

1. **Fit test:** the forest at T1 is cut into 30 risk classes by a map, the *calibration map*. Each class gets the share of its forest lost in 2014–2016. Those rates, spread over the cells, are compared with where the 2014–2016 loss happened.
2. **Prediction test:** the forest at T2 is cut into classes by the map *updated to T2*. Each class gets the rate learned in step 1, scaled to the total loss of 2017–2019, and the allocation is compared with where the 2017–2019 loss happened.

The map of step 2 is the same in both options. **The options differ only in the calibration map of step 1:**

| | Calibration map (fit test, and the rates carried to the prediction) | Used by |
|---|---|---|
| **A. Same timing** | Made from what was known at T1. The improved model: its predictors at the end of 2013. riskmapjnr: the local loss rate of 2011–2013. The benchmark: distance from non-forest at T1 | Every model in notebook 08's main table |
| **B. As designed** | riskmapjnr: the local loss rate of 2014–2016, the calibration window's own loss, which the fit test then checks | riskmapjnr's design (one row in notebook 08, section 4) |

Under B the fit test is close to circular: the map is made from the loss it is checked against. Section 1 explains what that means, with the numbers.

### In short

* **What the choice is about.** Verra's allocation gives each risk class the loss rate it had in the calibration window. The map those rates are learned on can be made from what was known at the start of the window (A), or from the window's own loss (B, riskmapjnr's design).
* **What it changes.** Which model has the lowest Verra errors, and so which model's map the platform uses. It does not change any model's AUC, nor the forest and loss areas.
* **With A,** the platform's risk map is the rank average of the improved model and riskmapjnr's local loss rate: Verra's prediction error 173 ha (deployed model 236 ha), AUC 0.759 (0.676).
* **With B,** the lowest prediction error would be riskmapjnr's own method (142 ha). The improved model would no longer allocate the loss, and Verra's fit test would check almost nothing, because the map is made from the loss it is scored on (section 1).
* **Option A is adopted.** It is the procedure of the tool in force, VT0007 v1.0 (§5.1: "the values of any dynamic variables … must be updated before the model is applied for prediction"; §5.3.1.1: "use distance from non-forest at T1"), and of the software written with Verra for it, UDef-ARP ("maps of covariates appropriate to T1", updated for T2). Option B comes from Verra's 2021 draft tool, which v1.0 replaced. riskmapjnr as designed is reported beside the adopted map as a sensitivity (sections 2 and 5).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent / "scripts"))
import model as MM
import calibration_timing as CT
import improve as I

OUT = HERE.parent / "outputs" / "summary"
RERUN = False
runs = [MM.cached(CT.split_name(s), lambda s=s: CT.run_split(s), rerun=RERUN) for s in MM.SPLITS]
WINDOWS = [run["split"].split(" → ")[1] for run in runs]
rows = []
for w, run in zip(WINDOWS, runs):
    for size in CT.SIZES_HA:
        b = run["blocks"][size]
        for key, m in run["models"].items():
            for test in ("fit", "prediction"):
                rows.append({"Block size": size, "Model": CT.LABELS[key], "Test": test, "Test window": w,
                             **CT.scores(m[size][test], b[f"observed_{test}"], b[f"forest_{test}"])})
scores = pd.DataFrame(rows)
scores.round(4).to_csv(OUT / "10_scores.csv", index=False)
METRICS = ["MedAE (ha)", "Mean absolute error (ha)", "wRMSE (ha)", "R²", "Misplaced (% of the loss)"]
A = [CT.LABELS[k] for k in ("benchmark", "improved_A", "rmj_A", "rank_A")]
B = [CT.LABELS[k] for k in ("rmj_B", "rmj_split_B", "rank_B")]

## 1 · What "close to circular" means

Verra's **fit test** asks: does the calibration map put the calibration window's loss in the right places? The test is only informative if the map was made without seeing that loss.

**Option A, the first window.** riskmapjnr colours each forest cell by how much forest was lost around it in 2011–2013. The fit test then checks that map against the loss of 2014–2016. That is a real test: it asks whether recent loss tells you where the next three years' loss will be. Sometimes it does, sometimes it doesn't, and the error shows it.

**Option B, the same window.** riskmapjnr colours each forest cell by how much forest was lost around it in **2014–2016**. The fit test then checks that map against the loss of **2014–2016**. Every cell that lost forest in 2014–2016 has, by construction, a high local loss rate, so it lands in the top classes. The map cannot miss: it is drawn from the answer. This is like scoring a weather forecast against the rain that was used to make it.

That is what "close to circular" means. It is not fully circular, because the map is a 5 km moving average rather than the loss itself, so a little error is left.

Two things follow:

* **The fit test stops telling models apart.** Under B, riskmapjnr's fit error is a fraction of any model that has to forecast (the table below). The benchmark and the improved model cannot get there, because their maps are made at T1.
* **The class rates it learns are steeper.** Under B the top classes hold the cells that were just cleared, so their learned loss rates are high and the bottom classes' rates are near zero. Those rates are what the prediction test carries forward. The prediction test itself is fair (it is scored on the next window's loss, which no map has seen), so B's prediction lead is real. It comes from a calibration step that VT0007 v1.0 does not describe.

In [2]:
CALIBRATION = [run["split"].split(" → ")[0] for run in runs]
fit = pd.DataFrame({c: {CT.LABELS[k]: run["models"][k]["fit_medae_ha"] for k in ("benchmark", "improved_A", "rmj_A", "rmj_B")}
                    for c, run in zip(CALIBRATION, runs)})
fit["Mean"] = fit.mean(axis=1)
print("Fit test: Verra's MedAE (ha), ~100,000 ha blocks, by calibration window")
fit.round(0)

Fit test: Verra's MedAE (ha), ~100,000 ha blocks, by calibration window


,2014–2016,2017–2019,2019–2021,Mean
VT0007 benchmark,252.0,385.0,422.0,353.0
Improved model (A),108.0,150.0,128.0,128.0
"riskmapjnr, same timing (A)",232.0,124.0,137.0,164.0
riskmapjnr as designed (B),18.0,27.0,21.0,22.0


**Reading:**

* **Under B, riskmapjnr's fit error is 18–27 ha per block, against 124–232 ha under A.** That is about 7 times lower on average (22 against 164 ha), from the timing of the map alone.
* **No model with a map made at T1 comes close.** The improved model's fit error is 108–150 ha and the benchmark's 252–422 ha. A fit error of ~20 ha in blocks of 100,000 ha says that the map saw the loss, not that it forecasts well.

## 2 · What the sources say

| Source | What it says | Supports |
|---|---|---|
| **Verra, VT0007 v1.0 (2024), §5.1**: the tool in force, which VM0048 and the JNR framework use | "The model used in the prediction phase is the same as for the fitting phase except that the values of any dynamic variables – maps that change over time – included in the model must be updated before the model is applied for prediction." | A |
| **VT0007 v1.0, §5.3.1.1**, the benchmark | "In the fitting phase of both the testing and application stages, use distance from non-forest at T1. For the prediction phase, this variable will have changed and must be updated." Alternative models follow the same steps with their own vulnerability map (§5.4.2), and must beat the benchmark "for both the fitting and prediction phases, using data from the calibration and confirmation periods respectively" (§5.5) | A |
| **Clark Labs, UDef-ARP tutorial**: the software written with Verra for VT0007 v1.0 (Eastman and Pontius, Clark University) | "In the CAL, a map of deforestation from time T1 to time T2 is used as the dependent variable. Together with maps of covariates appropriate to T1, models are empirically developed and compared with a goodness of fit test. Then these models are applied to the CNF by updating the covariates for T2." Its worked example updates every dynamic covariate to the start of the confirmation period, including two relative-frequency covariates | A, explicitly |
| **Verra, JNR Risk Mapping Tool, draft for public consultation (2021)**, labelled "VT0007, draft" | "The risk maps to be evaluated and compared shall be created with data from the calibration period only." Its risk model is the moving window over the recent past's loss. Only a prediction test (wRMSE), no fit test | B |
| **riskmapjnr and the deforisk QGIS plugin** (Vieilledent, CIRAD and FAO) | riskmapjnr implements the 2021 draft. deforisk scores the same maps with VT0007 v1.0's two tests. Vieilledent & Arsouze (2024): validation uses "independent observations of deforestation … (observed deforestation over the validation period have not be used to calibrate the model)" | B for the map, but on the independence of the test both agree |
| **The methodology document** (*Methodology_Deforestation_Guinea_v2*, §2) | Adopts VT0007 v1.0: "VT0007 requires that a candidate risk model be tested on a period distinct from the one it was fitted on" | VT0007 v1.0 |

**Reading:**

* **Option A is the procedure of the tool in force.** VT0007 v1.0 and Verra's own software build every model's fitting map from data at T1 and update it to T2 for the prediction. A moving-window loss rate is a dynamic variable like any other: at T1 it is the rate of the years before T1.
* **Option B comes from the 2021 draft that VT0007 v1.0 replaced** (the draft carried the same tool number). The draft had no fit test, so building the map from the calibration loss cost nothing there. VT0007 v1.0 added the fit test, and B makes that test nearly circular.
* **Nothing in VT0007 v1.0 forbids B in so many words** for alternative models, and the deforisk plugin applies v1.0's tests to B-built maps. So B is a reading some tools take, not the text's procedure. A verifier following VT0007 v1.0 and UDef-ARP would expect A.
* **One detail of the benchmark itself:** VT0007 takes its negligible-risk threshold from the whole historical period (T1–T3), which includes the confirmation window. Here the benchmark takes it from the calibration window only, which gives it no information from the test window.

## 3 · Does riskmapjnr's lead under B hold with other scores?

Verra's score is the median absolute error (MedAE) over blocks of about 100,000 ha. A median is set by the typical block, and most blocks have little loss, so it could favour a model that puts the loss where loss just happened. Other scores weigh the blocks differently:

* **Mean absolute error** and the **share of the loss misplaced** (half the sum of the absolute errors, over the total loss): every block counts by its error, the large ones too.
* **wRMSE:** the 2021 draft's score (riskmapjnr's), each block weighted by its forest; large errors count more.
* **R²** of allocated against observed loss, as deforisk reports it (higher is better).
* **Block size:** Verra's ~100,000 ha, and ~25,000 and ~400,000 ha.

The models: the benchmark; under A, the improved model, riskmapjnr and their rank average; under B, riskmapjnr as designed. One map per model (the first seed); one seed moves Verra's errors by 10–30 ha (notebook 05).

In [3]:
pred = scores[scores.Test == "prediction"]
mean = pred[pred["Block size"] == "100,000 ha"].groupby("Model", sort=False)[METRICS].mean()
print("Prediction test, ~100,000 ha blocks, mean of the three windows")
mean.loc[A + [CT.LABELS["rmj_B"]]].round(3)

Prediction test, ~100,000 ha blocks, mean of the three windows


,MedAE (ha),Mean absolute error (ha),wRMSE (ha),R²,Misplaced (% of the loss)
Model,,,,,
VT0007 benchmark,373.210,751.711,1608.341,0.767,17.559
Improved model (A),179.527,432.484,1000.267,0.911,9.903
"riskmapjnr, same timing (A)",193.347,398.417,924.951,0.924,9.011
Rank average (A),172.963,367.216,774.004,0.944,8.490
riskmapjnr as designed (B),141.910,317.639,759.977,0.953,7.482


In [4]:
# Which model is best on each score, block size and window (prediction test), among the A models and with B
best = []
for (size, w), g in pred.groupby(["Block size", "Test window"], sort=False):
    for metric in METRICS:
        pick = (lambda s: s.idxmax()) if metric == "R²" else (lambda s: s.idxmin())
        a = g[g.Model.isin(A[1:])].set_index("Model")[metric]
        ab = g[g.Model.isin(A[1:] + [CT.LABELS["rmj_B"]])].set_index("Model")[metric]
        best.append({"Block size": size, "Test window": w, "Score": metric, "Best under A": pick(a), "Best with B allowed": pick(ab)})
best = pd.DataFrame(best)
count = lambda col: best[col].value_counts().rename("Cases won (of 45)")
print("Prediction test: 5 scores x 3 block sizes x 3 windows = 45 cases")
display(count("Best under A").to_frame())
count("Best with B allowed").to_frame()

Prediction test: 5 scores x 3 block sizes x 3 windows = 45 cases


,Cases won (of 45)
Best under A,
Rank average (A),24
"riskmapjnr, same timing (A)",19
Improved model (A),2


,Cases won (of 45)
Best with B allowed,
riskmapjnr as designed (B),28
"riskmapjnr, same timing (A)",13
Rank average (A),4


In [5]:
key = ["MedAE (ha)", "Misplaced (% of the loss)"]
view = pred[pred["Block size"] == "100,000 ha"].pivot_table(index="Model", columns="Test window", values=key, sort=False)
print("Prediction test, ~100,000 ha, by window")
view.loc[A + [CT.LABELS["rmj_B"]]].round(1)

Prediction test, ~100,000 ha, by window


MedAE (ha)                      \
Test window                  2017–2019 2020–2022 2022–2024   
Model                                                        
VT0007 benchmark                 445.6     376.3     297.7   
Improved model (A)               233.9     167.5     137.1   
riskmapjnr, same timing (A)      280.1     149.9     150.1   
Rank average (A)                 234.8     161.8     122.3   
riskmapjnr as designed (B)       187.6     121.1     117.1   

                            Misplaced (% of the loss)                      
Test window                                 2017–2019 2020–2022 2022–2024  
Model                                                                      
VT0007 benchmark                                 17.9      18.0      16.8  
Improved model (A)                               11.9       9.5       8.2  
riskmapjnr, same timing (A)                      12.1       7.5       7.4  
Rank average (A)                                  9.7       7.6       8.2  
riskmapjnr as designed (B)                        7.3       6.7       8.4

**Reading:**

* **Under B, riskmapjnr's lead holds on every score on average** (~100,000 ha blocks): Verra's error 142 ha, 7.5 % of the loss misplaced, R² 0.95. It wins 28 of the 45 cases. So it is not the median that creates its lead.
* **But B's lead fades in the latest window.** In 2022–2024 it keeps only Verra's error at 100,000 ha (117 ha). On the scores that count every block, riskmapjnr under A does better (7.4 % of the loss misplaced against 8.4 %).
* **Under A, the improved model alone leads only on Verra's own score at 100,000 ha** (180 ha against 193 for riskmapjnr), and wins 2 of the 45 cases. On the scores that count every block, riskmapjnr's local rate does better in the last two windows (misplaced 7.5 % against 9.5 %, then 7.4 % against 8.2 %). The improved model misplaces more of the loss in the blocks that lose the most, which a median does not see.
* **The rank average of the two is the best model under A:** it wins 24 of the 45 cases, has the lowest Verra error (173 ha), and misplaces 8.5 % of the loss on average, against 9.9 % for the improved model alone and 9.0 % for riskmapjnr. Its AUC is the best of all models too (0.759, notebook 08).

## 4 · Can the improved model match option B?

If B were chosen, the improved model would need a calibration map with the same timing. Three ways, against riskmapjnr as designed:

* **riskmapjnr's classes split by our risk:** its 30 classes taken in pairs, each pair split in two by the improved model's risk (notebook 09).
* **Rank average under B:** the improved model's ranking averaged with riskmapjnr's local rate of the calibration window's own loss.
* **The improved model's own map at the end of the calibration window:** its predictors at T2 over the forest at T1 (notebook 09, three seeds).

In [6]:
fit_at_end = [MM.cached(I.split_name(s, "improve_fit_at_end"), lambda s=s: I.run_fit_at_end(s)) for s in MM.SPLITS]
rows = {}
for w, run, e in zip(WINDOWS, runs, fit_at_end):
    for k in ("rmj_B", "rmj_split_B", "rank_B", "improved_A"):
        rows.setdefault(CT.LABELS[k], {})[w] = run["models"][k]["prediction_medae_ha"]
    rows.setdefault("Improved model, its own map at the end of the window (B)", {})[w] = np.mean(e["geometric"]["prediction"])
b_table = pd.DataFrame(rows).T
b_table["Mean"] = b_table.mean(axis=1)
b_table.round(1).to_csv(OUT / "10_option_b_models.csv")
print("Prediction MedAE (ha), ~100,000 ha blocks")
b_table.round(1)

Prediction MedAE (ha), ~100,000 ha blocks


,2017–2019,2020–2022,2022–2024,Mean
riskmapjnr as designed (B),187.6,121.0,117.1,141.9
riskmapjnr's classes split by our risk (B),200.5,125.3,149.4,158.4
"Rank average, riskmapjnr as designed (B)",203.9,142.5,169.6,172.0
Improved model (A),233.9,167.5,137.1,179.5
"Improved model, its own map at the end of the window (B)",382.7,210.7,204.8,266.1


**Reading:**

* **None of the three ways beats riskmapjnr as designed.** Splitting its classes by our risk comes closest (158 ha against 142). The rank average under B (172 ha) does no better than under A (173 ha).
* **The improved model's own map at the end of the window is far worse** (266 ha): each cell cleared in the window carries its own loss in its distance to recent loss, so the top classes learn rates near one.
* **So if B were chosen, the platform's model would be riskmapjnr itself**, or its classes split by our risk.

## 5 · What this means

* **Why A rather than B.** Three reasons, in order of weight:
  1. **It is what the tool in force says.** VT0007 v1.0 (§5.1, §5.3.1.1) and Verra's own software for it (UDef-ARP) build every model's fitting map from data at T1, then update it to T2 for the prediction.
  2. **B comes from a superseded draft.** It is the procedure of Verra's 2021 draft tool, which carried the same number (VT0007) and was replaced by v1.0 in 2024. The draft had no fit test, so the timing of the map did not matter there.
  3. **Under B, v1.0's fit test checks almost nothing** (section 1): riskmapjnr's map is drawn from the loss it is then scored on.

  B is not forbidden in so many words, and riskmapjnr's prediction under B is a fair test (the next window's loss is unseen). So B stays as a sensitivity, not as an error.
* **Under A, the best model is the rank average of the improved model and riskmapjnr's local loss rate**, not the improved model alone. It has the best Verra prediction error (173 ha) and the best AUC (0.759), and it wins 24 of the 45 score comparisons. It needs no new data: the 5 km local loss rate is already one of the improved model's predictors, and averaging two percentile ranks is a few lines of code.
* **Under B, riskmapjnr as designed leads** (142 ha), and no combination with our model beats it. Its lead is clearest in the earlier windows and gone in the latest one on the scores that count every block.
* **Adopted for the platform:**
  1. **Option A**, the procedure of VT0007 v1.0 and of its software, UDef-ARP.
  2. **The rank average as the platform's risk map.** Against the deployed model it lowers Verra's prediction error from 236 to 173 ha and raises the AUC from 0.676 to 0.759. Under the same timing it beats every other tool tested.
  3. **riskmapjnr as designed reported beside it**, as a sensitivity: under B it would lead (142 ha), and a verifier using the deforisk plugin could build it that way.
* **Limits:** one map per model (the first seed; Verra's errors move by 10–30 ha with the seed alone), three windows, and Hansen loss, most of which is a fallow cycle (dataset check 04).

### References

* Busch, J., Ferretti-Gallon, K. (2023). What drives and stops deforestation, reforestation, and forest degradation? An updated meta-analysis. *Review of Environmental Economics and Policy* 17(2), 217–250.
* Clark Labs (2024–2025). *UDef-ARP: Unplanned Deforestation Allocated Risk mapping Procedure*, software and tutorial (`doc/UDEF-ARP_Tutorial.pdf`). https://github.com/ClarkCGA/UDef-ARP
* Pontius, R.G. Jr., Si, K. (2014). The total operating characteristic to measure diagnostic ability for multiple thresholds. *International Journal of Geographical Information Science* 28(3), 570–583.
* Verra (2021). *VT0007 JNR Risk Mapping Tool, draft v0.1 for public consultation*, 15 April 2021.
* Verra (2024). *VT0007 Unplanned Deforestation Allocation (UDef-A), v1.0*, 21 February 2024.
* Vieilledent, G., Arsouze, T. (2024). *Jurisdictional risk maps for allocating deforestation*. FAO workshop, Santa Marta, July 2024. https://ecology.ghislainv.fr/presentations/riskmaps-verra-July2024.pdf
* Vieilledent, G. (2021–2025). *riskmapjnr* and *deforisk* (QGIS plugin). https://deforisk-qgis-plugin.org